### **Cuaderno de verificación**

Notebook de verificación del entorno del curso. Ejecuta todas las celdas antes de empezar el trabajo semanal.

In [ ]:
# Celda 1: versiones
import sys
import importlib

print("Python:", sys.version)
modules = [
    "numpy", "pandas", "matplotlib", "sklearn", "torch",
    "transformers", "datasets", "nltk", "spacy", "evaluate"
]
for m in modules:
    mod = importlib.import_module(m)
    version = getattr(mod, "__version__", "sin atributo __version__")
    print(f"{m}: {version}")

In [ ]:
# Celda 2: torch
import torch

print("Torch version:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("Tensor OK:", x.mean().item())

In [ ]:
# Celda 3: tokenizer HF
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased",
    clean_up_tokenization_spaces=False
)

sample = tokenizer("Verificación rápida del entorno de NLP.", return_tensors="pt")
print(sample["input_ids"].shape)

In [ ]:
# Celda 4: datasets HF
import os
import requests
from datasets import load_dataset

os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

print("homepage:", requests.get("https://huggingface.co", timeout=30).status_code)
print("dataset api:", requests.get("https://huggingface.co/api/datasets/ag_news", timeout=30).status_code)

ds = load_dataset("ag_news", split="train[:5]")
print(ds)
print(ds[0])

In [ ]:
# Celda 5: nltk
import nltk
from nltk.tokenize import word_tokenize

nltk.download("punkt_tab")

text = "Este es un ejemplo corto para validar NLTK."
print(word_tokenize(text))

In [ ]:
#import sys
#!{sys.executable} -m spacy download es_core_news_sm

In [ ]:
# Celda 6: spaCy
import spacy

try:
    nlp = spacy.load("es_core_news_sm")
except OSError:
    raise RuntimeError(
        "No se encontró el modelo es_core_news_sm. "
        "Instálalo en la imagen o ejecuta: python -m spacy download es_core_news_sm"
    )

doc = nlp("La evaluación del curso exige comprensión conceptual y reproducibilidad.")
print([(t.text, t.pos_) for t in doc[:10]])

In [ ]:
# Celda 7:faiss
import numpy as np
import faiss

np.random.seed(42)

xb = np.random.random((100, 16)).astype("float32")
xq = np.random.random((3, 16)).astype("float32")

index = faiss.IndexFlatL2(16)
index.add(xb)

D, I = index.search(xq, k=5)

print("Vectores indexados:", index.ntotal)
print("Índices vecinos más cercanos:")
print(I)
print("Distancias:")
print(D)

In [ ]:
# Celda 8: peft
from transformers import BertConfig, BertForSequenceClassification
from peft import LoraConfig, get_peft_model

config = BertConfig(
    vocab_size=30522,
    hidden_size=64,
    num_hidden_layers=2,
    num_attention_heads=2,
    intermediate_size=128,
    num_labels=2
)

base_model = BertForSequenceClassification(config)

lora_config = LoraConfig(
    task_type="SEQ_CLS",
    r=4,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

peft_model = get_peft_model(base_model, lora_config)

peft_model.print_trainable_parameters()
print("Modelo PEFT creado correctamente.")

In [ ]:
# Celda 9: sentence-transformers
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("sentence-transformers/paraphrase-MiniLM-L3-v2")

sentences = [
    "El procesamiento de lenguaje natural es interesante.",
    "NLP es un campo muy útil en inteligencia artificial.",
    "Me gusta cocinar pasta los domingos."
]

embeddings = model.encode(sentences, convert_to_tensor=True)
sim = util.cos_sim(embeddings, embeddings)

print("Embeddings shape:", embeddings.shape)
print("Matriz de similitud:")
print(sim)

In [ ]:
#Celda 10:diffusers
import torch
from diffusers import DDPMScheduler

scheduler = DDPMScheduler(num_train_timesteps=1000)

sample = torch.randn(1, 3, 32, 32)
noise = torch.randn_like(sample)
timesteps = torch.tensor([10])

noisy_sample = scheduler.add_noise(sample, noise, timesteps)

print("Sample original:", sample.shape)
print("Sample con ruido:", noisy_sample.shape)
print("Prueba de diffusers OK")

In [ ]:
# Celda 11: gradio
import gradio as gr

def saludar(nombre):
    return f"Hola, {nombre}"

demo = gr.Interface(fn=saludar, inputs="text", outputs="text")
print("Interfaz Gradio creada correctamente:", type(demo).__name__)

In [ ]:
# Celda 12: streamlit
import streamlit as st
print("Streamlit OK:", st.__version__)

#### **Criterio de aprobación**

El entorno queda verificado si:

- todos los imports cargan
- `torch` funciona en CPU
- se descarga un tokenizer
- se carga un dataset pequeño
- NLTK y spaCy responden sin error.